# 01 OpenICU → YAIB dynamic table

Build `openicu_dyn.parquet` from OpenICU MEDS-like concept parquets.

In [ ]:
from pathlib import Path
import polars as pl

from openicu_yaib.transform import build_dynamic_table
from openicu_yaib.concepts import DYNAMIC_VARS

## Paths

Adjust these paths for your machine.

In [ ]:
CONCEPT_ROOT = Path("~/workspace/OpenICU.example/output/project/workspace/concept")
ICUSTAYS_CSV = Path("~/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz")
RICU_CONCEPT_DICT = Path("~/workspace/ricu/inst/extdata/config/concept-dict.json")
OUTPUT = Path("~/output/openicu_yaib/openicu_dyn.parquet")

DATASET = "mimic-iv"
VERSION = "1.0.0"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT

## Build dynamic table

`grid_end_rounding="floor"` is currently used because it matched the observed RICU `stay_windows()` end values better than `ceil`.

In [ ]:
lf = build_dynamic_table(
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    dataset=DATASET,
    version=VERSION,
    dynamic_vars=DYNAMIC_VARS,
    aggregation_mode="mean",
    include_grid=True,
    max_hours=168,
    grid_end_rounding="floor",
    filter_to_icu_window=True,
    missing_concepts="warn",
)

lf.sink_parquet(OUTPUT)
OUTPUT

## Quick check

In [ ]:
df = pl.scan_parquet(OUTPUT)

df.select([
    pl.len().alias("n_rows"),
    pl.col("stay_id").n_unique().alias("n_stays"),
    pl.col("time").min().alias("min_time"),
    pl.col("time").max().alias("max_time"),
]).collect()

In [ ]:
df.collect()

In [ ]:
import polars as pl
from pathlib import Path

output_path = Path("~/output/openicu_yaib")

openicu = pl.read_parquet(output_path / "openicu_dyn.parquet")

openicu_windows = (
    openicu
    .group_by("stay_id")
    .agg([
        pl.col("time").min().alias("openicu_start"),
        pl.col("time").max().alias("openicu_end"),
        pl.col("time").n_unique().alias("openicu_n_timepoints"),
    ])
    .with_columns(
        (pl.col("openicu_end") - pl.col("openicu_start") + 1)
        .alias("openicu_expected_n_timepoints")
    )
    .with_columns(
        (pl.col("openicu_n_timepoints") - pl.col("openicu_expected_n_timepoints"))
        .alias("openicu_missing_grid_points")
    )
    .sort("stay_id")
)

openicu_windows

In [ ]:
ricu_stay_windows = pl.read_parquet(output_path / "ricu_stay_windows_miiv.parquet")

ricu_windows = (
    ricu_stay_windows
    .with_columns([
        (pl.col("start").dt.total_seconds() / 3600)
        .cast(pl.Int64)
        .alias("ricu_start"),

        (pl.col("end").dt.total_seconds() / 3600)
        .cast(pl.Int64)
        .alias("ricu_end"),
    ])
    .select([
        pl.col("stay_id").cast(pl.Int64),
        "ricu_start",
        "ricu_end",
    ])
    .with_columns(
        (pl.col("ricu_end") - pl.col("ricu_start") + 1)
        .alias("ricu_expected_n_timepoints")
    )
    .sort("stay_id")
)

ricu_windows

In [ ]:
window_compare = (
    openicu_windows
    .join(
        ricu_windows,
        on="stay_id",
        how="outer",
        coalesce=True,
    )
    .with_columns([
        (pl.col("openicu_start") - pl.col("ricu_start")).alias("diff_start"),
        (pl.col("openicu_end") - pl.col("ricu_end")).alias("diff_end"),
        (pl.col("openicu_n_timepoints") - pl.col("ricu_expected_n_timepoints")).alias("diff_n_timepoints"),
    ])
    .sort("diff_end")
)

window_compare

In [ ]:
window_compare.select([
    pl.len().alias("n_stays_total"),

    pl.col("openicu_start").is_not_null().sum().alias("n_stays_in_openicu"),
    pl.col("ricu_start").is_not_null().sum().alias("n_stays_in_ricu"),

    (pl.col("diff_start") == 0).sum().alias("n_same_start"),
    (pl.col("diff_end") == 0).sum().alias("n_same_end"),

    (pl.col("diff_end") < 0).sum().alias("n_openicu_shorter"),
    (pl.col("diff_end") > 0).sum().alias("n_openicu_longer"),

    pl.col("diff_end").min().alias("min_diff_end"),
    pl.col("diff_end").max().alias("max_diff_end"),
    pl.col("diff_end").mean().alias("mean_diff_end"),
])

In [ ]:
window_compare.filter(
    pl.col("diff_end") != 0
).select([
    "stay_id",
    "openicu_start",
    "openicu_end",
    "openicu_n_timepoints",
    "ricu_start",
    "ricu_end",
    "ricu_expected_n_timepoints",
    "diff_start",
    "diff_end",
    "diff_n_timepoints",
]).sort("diff_end")